# 14 · Project Insight (capstone)
*Raw feed to governed, monitored briefings, with a runbook for 3 a.m.*

Everything you built comes together. This notebook is the **health check**: the last task of the `shield_daily` job. It only reads. It checks that the layers still add up,
and when something looks wrong it raises an error, so the job fails and your alert fires. A green job that quietly serves bad data is worse than a red one.
Read [docs/runbook.md](../../docs/runbook.md) for what to do when it fails.

| Concept | Where you meet it |
|---|---|
| A health check as the last job task: fail loudly | the next section, Step 6 |
| Reconciliation: nothing lost between layers | Step 1 |
| A quality rate with a tolerance | Step 2 |
| Freshness | Step 3 |
| Uniqueness of the silver key | Step 4 |
| System tables for cost and job history | Step 5 |

Deep dive: [docs/14-project-insight.md](../../docs/14-project-insight.md)

**Needs** (it creates nothing, but it cannot check what does not exist):
* `shield.bronze.incident_reports` (chapter 03 or the `ingest` task), `shield.silver.incident_reports` and `shield.silver.incident_reports_quarantine` (chapter 06 or the `silver` task), and `shield.gold.daily_threat_briefing` (chapter 05, 06 or the `gold` task). If one is missing, the Setup cell stops with a clear message. That also fails the job, but it means "could not check", not "found a problem": the runbook tells the two apart.
* Step 5 is optional: it needs system tables (a metastore admin has to enable the `lakeflow` schema and grant you `SELECT`). Without them it prints a message and moves on.

**How the job calls it:** [`resources/shield_job.yml`](../../resources/shield_job.yml) defines the task `health_check`, which runs this notebook after `gold` and passes **no parameters**, so the notebook has no widgets. The job fails if the notebook raises.
The job file names no compute, so the task relies on serverless compute for jobs being enabled in your workspace. Running the job costs DBUs, and its schedule is deployed **paused**.

The **Expected** numbers assume the dataset from chapter 02 (10,000 incidents, seed 42, 5 % dirty) with all five files in bronze; other data gives other numbers and a ❌, which is not a bug.
Inside the job, `ingest` may deliver only some of the files per run. The checks compare layers with each other, so they still pass; only the printed counts differ.

## The health check

**The idea:** a *health check* is a set of invariants that must hold after every run, evaluated by code, with a failure that someone sees. It is a pattern, not a Databricks feature: you pick the invariants. Databricks also ships built-in data quality monitoring, which this repo does not cover.

```
 ingest ──▶ silver ──▶ gold ──▶ health_check      reads bronze, silver, quarantine, gold. Writes nothing.
                                    │
                    ┌───────────────┴───────────────┐
              all checks pass                  any check fails
              task succeeds                    RuntimeError → task fails → the job's failure notification
```

| Check | The question | Passes when | Step |
|---|---|---|---|
| `reconciliation` | did any incident vanish between bronze and silver plus quarantine? | silver + quarantined incidents ≥ 99 % of the distinct bronze incidents | 1 |
| `quarantine_rate` | is a producer sending much more garbage than usual? | under 15 % of incidents are quarantined | 2 |
| `gold_freshness` | did the gold task actually write? | the latest change to the gold table is under 24 hours old | 3 |
| `silver_unique` | did duplicates leak into silver? | no `incident_id` appears twice | 4 |

**How this notebook implements it**

* One small `check(name, ok, detail)` function (Setup) records each result in a list and shows it as a one-row table. Every step is one check. The thresholds are named constants at the top of Setup.
* The checks never raise by themselves. The last step collects the failures and raises **once**, with all the names, so one run tells you everything that is wrong.
* Step 5 (cost and job history) is information for humans and never fails the task.

**Why it helps:** the check runs at the end of every job, so a bad feed shows up as a red task within minutes. You can tell which layer broke from the check name, and the runbook maps each name to a first action.

**Honest limits:** the thresholds (99 %, 15 %, 24 hours) are choices for this synthetic data, not rules. The checks do not look inside values, and they say nothing about schema drift (mini-project 3 in the doc). Alert routing (who gets the email) was set up in chapter 10.

## Setup

Run the cells top to bottom. Running Setup again clears the collected results.

In [ ]:
from pyspark.sql import functions as F

RECONCILIATION_TOLERANCE = 0.99  # silver + quarantine must cover at least this share of the bronze incidents
QUARANTINE_RATE_LIMIT = 0.15  # fail when more than this share of incidents is quarantined
FRESHNESS_LIMIT_HOURS = 24  # fail when gold has not changed for this long

problems: list[str] = []
health_results: list[tuple[str, str, str]] = []


def check(name: str, ok: bool, detail: str) -> None:
    """Record one health check, show it as a one-row table, and remember its name when it failed."""
    status = "✅ PASS" if ok else "❌ FAIL"
    health_results.append((name, status, detail))
    if not ok:
        problems.append(name)
    display(spark.createDataFrame([(name, status, detail)], "check string, status string, detail string"))


needed = [
    "shield.bronze.incident_reports",
    "shield.silver.incident_reports",
    "shield.silver.incident_reports_quarantine",
    "shield.gold.daily_threat_briefing",
]
missing = [t for t in needed if not spark.catalog.tableExists(t)]
assert not missing, f"Cannot run the health check, run notebooks 03 and 06 (or the shield_daily job) first. Missing tables: {missing}"

## 1 · Layers

### Step 1 · Reconciliation: nothing lost between layers
**Goal:** every incident that arrived in bronze ended up in silver or in quarantine.

| Input | What is counted |
|---|---|
| `shield.bronze.incident_reports` | distinct `incident_id` (bronze keeps replayed duplicates) |
| `shield.silver.incident_reports` | rows (silver has one row per incident) |
| `shield.silver.incident_reports_quarantine` | distinct `incident_id` (a bad incident can be quarantined more than once) |

**Expected:** with the chapter-02 dataset and all five files, bronze has 10,000 distinct incidents, silver 9,500 and quarantine 500, so the sum is exactly 10,000 and the check passes.
The 1 % slack in `RECONCILIATION_TOLERANCE` is a choice: tighten it to 1.0 if you want an exact match.

| check | status | detail |
|---|---|---|
| reconciliation | ✅ PASS | bronze=10000 silver=9500 quarantine=500 |

Read-only, so it is safe to re-run.

In [ ]:
bronze_incident_count = spark.table("shield.bronze.incident_reports").select("incident_id").distinct().count()
silver_incident_count = spark.table("shield.silver.incident_reports").count()
quarantined_incident_count = spark.table("shield.silver.incident_reports_quarantine").select("incident_id").distinct().count()

check(
    "reconciliation",
    silver_incident_count + quarantined_incident_count >= bronze_incident_count * RECONCILIATION_TOLERANCE,
    f"bronze={bronze_incident_count} silver={silver_incident_count} quarantine={quarantined_incident_count}",
)

### Step 2 · Quality: quarantine rate within tolerance
**Goal:** notice when a producer starts sending much more bad data than usual.

**Expected:** the generator makes 5 % of incidents defective, so 500 of 10,000 incidents (5.0 %) are quarantined, well under the 15 % limit. The cell reuses the counts from Step 1, so run Step 1 first.

| check | status | detail |
|---|---|---|
| quarantine_rate | ✅ PASS | 5.0% of incidents quarantined (limit 15%) |

To rehearse a failure without touching any data, set `QUARANTINE_RATE_LIMIT = 0.01` in Setup and run again: 5.0 % is above 1 %, so this check fails and Step 6 raises. Put the limit back to 0.15 afterwards.

In [ ]:
quarantine_rate = quarantined_incident_count / max(bronze_incident_count, 1)
check(
    "quarantine_rate",
    quarantine_rate < QUARANTINE_RATE_LIMIT,
    f"{quarantine_rate:.1%} of incidents quarantined (limit {QUARANTINE_RATE_LIMIT:.0%})",
)

## 2 · Time and keys

### Step 3 · Freshness: gold was updated recently
**Goal:** catch a pipeline that ran green but did not write, or did not run at all.

**Input:** the Delta history of `shield.gold.daily_threat_briefing` (chapter 07). Any operation counts as a write, including a maintenance run.

**Expected:** right after the `gold` task, the age is a few minutes or less and the check passes. If you run this notebook by hand days after chapter 06, it **fails** on purpose: the data really is old, and that is the check working. In that case run the `gold` step (chapter 10) and try again.

| check | status | detail |
|---|---|---|
| gold_freshness | ✅ PASS | gold last written 0.1 hours ago (limit 24) |

The age is computed in Spark, from the current timestamp and the last write, so no time zone conversion happens in Python.

In [ ]:
gold_age_hours = (
    spark.sql("DESCRIBE HISTORY shield.gold.daily_threat_briefing")
    .agg(F.max("timestamp").alias("last_write"))
    .select(((F.unix_timestamp(F.current_timestamp()) - F.unix_timestamp("last_write")) / 3600).alias("age_hours"))
    .first()["age_hours"]
)
check(
    "gold_freshness",
    gold_age_hours < FRESHNESS_LIMIT_HOURS,
    f"gold last written {gold_age_hours:.1f} hours ago (limit {FRESHNESS_LIMIT_HOURS})",
)

### Step 4 · Uniqueness: no duplicate keys in silver
**Goal:** make sure the deduplication of silver still works, so no incident is counted twice in gold.

**Expected:** 0 duplicated `incident_id` values. Silver is built with one row per incident (chapters 05, 06 and 10), so a non-zero number means that step broke.

| check | status | detail |
|---|---|---|
| silver_unique | ✅ PASS | 0 duplicated incident_ids |

In [ ]:
duplicated_incident_ids = spark.sql(
    "SELECT COUNT(*) FROM (SELECT incident_id FROM shield.silver.incident_reports GROUP BY 1 HAVING COUNT(*) > 1)"
).first()[0]
check("silver_unique", duplicated_incident_ids == 0, f"{duplicated_incident_ids} duplicated incident_ids")

## 3 · Cost and operations

### System tables

**The idea:** *system tables* are read-only tables that Databricks maintains for you in the `system` catalog. They hold your account's operational data, so you can query usage and job history with SQL like any other table. They are a feature, not a pattern.

| Table | One row is | Used here for |
|---|---|---|
| `system.billing.usage` | usage of one SKU in a time slice, in DBUs (not money; prices are in `system.billing.list_prices`) | DBUs per day and SKU, last 14 days |
| `system.lakeflow.job_run_timeline` | a slice of a job run, up to one hour long; `result_state` is filled only on the last slice | runs per job and outcome, last 7 days |

**How this notebook implements it:** two read-only queries inside one `try`. If the schemas are not enabled or you lack `SELECT`, the cell prints a message and the task still succeeds. Cost is information here, not a gate.

**Why it helps:** when the bill jumps, or a job fails every Monday, the answer is one query away, and a health check is a natural place to print it.

**Honest limits:** billing records can lag by hours (up to about 12 according to the docs), and job records by about an hour, so the run that is executing right now does not show up yet. Turning DBUs into money needs `list_prices`, which this notebook does not do.

### Step 5 · Print DBUs per day and job outcomes
**Goal:** see what the platform costs and how its jobs end, without leaving the health check.

**Expected:** two tables, or the line `System tables unavailable ...`. Both are fine. With data, the first table lists up to 20 rows of (date, SKU, DBUs) and the second the number of runs per job and result (`SUCCEEDED`, `FAILED`, ...). The numbers depend on your workspace, so there is no check here.

The job-history query counts distinct runs with a final `result_state`, so a long run split into hourly slices is counted once. The first table is capped at 20 rows on purpose, to keep the task output short.

In [ ]:
try:
    display(
        spark.sql(
            """
            SELECT usage_date, sku_name, ROUND(SUM(usage_quantity), 2) AS dbus
            FROM system.billing.usage
            WHERE usage_date >= current_date() - INTERVAL 14 DAYS
            GROUP BY ALL ORDER BY usage_date DESC, dbus DESC LIMIT 20
            """
        )
    )
    display(
        spark.sql(
            """
            SELECT job_id, result_state, COUNT(DISTINCT run_id) AS runs
            FROM system.lakeflow.job_run_timeline
            WHERE period_start_time >= current_timestamp() - INTERVAL 7 DAYS
              AND result_state IS NOT NULL
            GROUP BY ALL ORDER BY runs DESC LIMIT 20
            """
        )
    )
except Exception as error:  # noqa: BLE001
    print("System tables unavailable (enable the schemas and grant SELECT first):", type(error).__name__)

## 4 · The verdict

### Step 6 · Fail the task when a check failed
**Goal:** turn the collected checks into the task's status, so the job's failure notification fires.

**Expected:** on a healthy platform, a summary table with four ✅ rows and the line `All checks passed.` If any check failed, the cell shows the summary and then raises `RuntimeError: Project Insight health check failed: [...]` with the names of the failed checks. In a job, that turns the `health_check` task red; run by hand, the cell turns red.

| check | status | detail |
|---|---|---|
| reconciliation | ✅ PASS | bronze=10000 silver=9500 quarantine=500 |
| quarantine_rate | ✅ PASS | 5.0% of incidents quarantined (limit 15%) |
| gold_freshness | ✅ PASS | gold last written 0.1 hours ago (limit 24) |
| silver_unique | ✅ PASS | 0 duplicated incident_ids |

In [ ]:
display(spark.createDataFrame(health_results, "check string, status string, detail string"))

# Fail the task so the job's failure notification fires.
if problems:
    raise RuntimeError(f"Project Insight health check failed: {problems}")
print("All checks passed.")

## 👀 In the workspace

* **Jobs & Pipelines** → `shield_daily` → the latest run → the `health_check` task: the summary table is in its output, and a failed run shows the `RuntimeError` message with the check names.
* The job's **notifications** (chapter 10) are what turns a red `health_check` into an email. The job schedule is deployed paused, so nothing runs until you start it.
* **Catalog** → `system` (if enabled) → `billing` and `lakeflow`: the tables Step 5 queried.

## 🧹 Clean up

This notebook creates nothing, so there is nothing to delete. The platform itself does cost money while it exists: compute and Event Hubs keep billing. When you are finished with the whole series, tear it down with `azd down --purge` (chapters 00 and 01), and remove bundle-deployed jobs with `databricks bundle destroy` (see the [deploy guide](../../docs/deploy-and-automate.md)).

## 🎯 Recap

* The health check reads four layers and judges them with four checks: reconciliation, quarantine rate, freshness, uniqueness. It raises one error listing every failure, so the job's alert fires.
* Cost and job history come from system tables, printed for humans and never a reason to fail.
* Every check name maps to a first action in [docs/runbook.md](../../docs/runbook.md), so a 3 a.m. responder starts from the failed task, not from guesswork.

Mini-projects: [docs/14-project-insight.md](../../docs/14-project-insight.md#try-it-yourself)

> 💡 That is the series: a platform that ingests, cleans, serves and watches itself. Next, pick a dataset of your own and rebuild it from the doc's checklist, then write your five-minute walkthrough.